# codingStandard — Google Colab Validation

Run this notebook from a clean Google Colab runtime. It validates environment detection, the LLM memory smoke test, the Vision memory smoke test, and repository validation.

If the repository is private, provide a GitHub token with read access to this repository. The notebook first checks the `GITHUB_TOKEN` Colab Secret/environment variable and otherwise asks for the token securely. The token is never written to the clone URL or notebook output.

The test is intentionally small. Passing means the standard's minimal execution paths work in the current Colab runtime; it does not guarantee that a production model will fit in the same runtime.

In [ ]:
from pathlib import Path
import getpass
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/eaglesjo/codingStandard.git'
REPO = Path('/content/codingStandard')
RESULTS = Path('/content/codingstandard-colab-results.json')

def _colab_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None

def _github_token():
    token = os.environ.get('GITHUB_TOKEN') or _colab_secret('GITHUB_TOKEN')
    if token:
        return token
    return getpass.getpass('GitHub token (leave blank for public repository): ').strip() or None

def clone_repository():
    if REPO.exists():
        if (REPO / '.git').is_dir():
            return
        shutil.rmtree(REPO)

    token = _github_token()
    env = os.environ.copy()
    askpass = None
    if token:
        askpass = Path('/tmp/codingstandard-git-askpass.sh')
        askpass.write_text(
            '#!/bin/sh\n'
            'case \"$1\" in\n'
            '  *Username*) echo x-access-token ;;\n'
            '  *) echo \"$GITHUB_TOKEN\" ;;\n'
            'esac\n',
            encoding='utf-8',
        )
        askpass.chmod(0o700)
        env['GITHUB_TOKEN'] = token
        env['GIT_ASKPASS'] = str(askpass)
        env['GIT_TERMINAL_PROMPT'] = '0'

    try:
        proc = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)],
            check=False,
            capture_output=True,
            text=True,
            env=env,
        )
        if proc.returncode != 0:
            detail = (proc.stderr or proc.stdout or 'unknown git clone error').strip()
            raise RuntimeError(f'Git clone failed (exit {proc.returncode}): {detail}')
    finally:
        if askpass:
            askpass.unlink(missing_ok=True)

clone_repository()
os.chdir(REPO)
print('Repository:', REPO)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())


In [ ]:
# Install only the lightweight dependency needed by the profiler when absent.
if importlib.util.find_spec('psutil') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil'], check=True)

if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch is not available. Enable a Colab runtime with PyTorch installed before running this test.')

import psutil
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB')
mem = psutil.virtual_memory()
print(f'RAM available: {mem.available / 1024**3:.2f} GB / {mem.total / 1024**3:.2f} GB')


In [ ]:
# Optional hardware diagnostic.
result = subprocess.run(['bash', '-lc', 'command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv,noheader || true'], capture_output=True, text=True)
print(result.stdout.strip() or 'nvidia-smi not available (CPU runtime or non-NVIDIA runtime).')


In [ ]:
# 1) Shared environment profiler
profile_path = Path('/content/colab-environment-profile.json')
subprocess.run([sys.executable, 'LLM/environment.py', str(profile_path)], check=True)
profile = json.loads(profile_path.read_text(encoding='utf-8'))
print(json.dumps(profile, indent=2, ensure_ascii=False))


In [ ]:
# 2) LLM memory smoke test
llm_json = Path('/content/colab-llm-smoke.json')
llm_cmd = [sys.executable, 'LLM/memory_smoke_test.py', '--steps', '2', '--batch-size', '1', '--min-ram-free-gb', '0.25', '--min-vram-free-gb', '0.10', '--json', str(llm_json)]
llm_run = subprocess.run(llm_cmd, capture_output=True, text=True)
print(llm_run.stdout)
if llm_run.returncode != 0:
    print(llm_run.stderr)
    raise RuntimeError('LLM memory smoke test failed')
llm_result = json.loads(llm_json.read_text(encoding='utf-8'))


In [ ]:
# 3) Vision memory smoke test with a deliberately small image tensor
vision_json = Path('/content/colab-vision-smoke.json')
vision_cmd = [sys.executable, 'VISION/memory_smoke_test.py', '--device', 'auto', '--image-size', '64', '--batch-size', '1', '--steps', '2', '--output', str(vision_json)]
vision_run = subprocess.run(vision_cmd, capture_output=True, text=True)
print(vision_run.stdout)
if vision_run.returncode != 0:
    print(vision_run.stderr)
    raise RuntimeError('Vision memory smoke test failed')
vision_result = json.loads(vision_json.read_text(encoding='utf-8'))


In [ ]:
# 4) Repository validation
subprocess.run([sys.executable, 'scripts/validate.py'], check=True)
print('Repository validation passed.')


In [ ]:
# Final result bundle
summary = {
    'platform': 'google_colab',
    'python': sys.version.split()[0],
    'pytorch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'environment_profile': str(profile_path),
    'llm_smoke_test': llm_result,
    'vision_smoke_test': vision_result,
    'status': 'passed',
}
RESULTS.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Saved:', RESULTS)
